In [1]:
import pickle 
import numpy as np 
import itertools
import pandas as pd
from pathlib import Path 


# Generate conditions for symmetric distractor test.

Test in style of Kidd / Marrone et al. (2008) / Srinivasan et al. (2016) experiments 

Target at 0 aziuth    
Distractors symmetrically presented at +/- 0, 5, 10, 20, and 40 degrees azimuth    
All signals at 0 elevation     
Left and right distractor level summed and set to 60dB SPL
- equal power at left and right for model first pass
- human experiments roved L/R balance slightly

Target set against distractor at desired SNR 

### Want to test effect of reverberation in this setting 
Use rooms at `/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/` which are versions of the speaker array room with alternative reverb profiles.

## This Notebook
Generate location x snr conditions to get model performance and thresholds, per room 

## Make manifests for new minimally reverberant simulation of MIT 46 1004

### Load manifest of rooms

In [2]:
new_min_rev_dir = Path("/om2/user/imgriff/spatial_audio_pipeline/assets/brir/mit_bldg46room1004_min_reverb")
room_manifest = pd.read_pickle(new_min_rev_dir / 'manifest_room.pdpkl')
room_manifest

,head_azim,head_pos_xyz,index_room,is_outdoor,material_x0,material_x1,material_y0,material_y1,material_z0,material_z1,room_dim_xyz,room_materials
0,0,"[2.3, 3.6, 0.9]",0,False,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,carpet on foam rubber padding,"highly absorptive panels, 1in, 16in below ceiling","[4.66, 5.9, 2.48]","[11, 11, 11, 11, 15, 20]"
1,0,"[3.6, 2.36, 0.9]",1,False,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,carpet on foam rubber padding,"highly absorptive panels, 1in, 16in below ceiling","[5.9, 4.66, 2.48]","[11, 11, 11, 11, 15, 20]"
2,0,"[2.36, 2.3, 0.9]",2,False,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,carpet on foam rubber padding,"highly absorptive panels, 1in, 16in below ceiling","[4.66, 5.9, 2.48]","[11, 11, 11, 11, 15, 20]"
3,0,"[2.3, 2.3, 0.9]",3,False,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,carpet on foam rubber padding,"highly absorptive panels, 1in, 16in below ceiling","[5.9, 4.66, 2.48]","[11, 11, 11, 11, 15, 20]"


### Emulation of actual human experiment

Symmetric distractor is true for azimuth and elevation - we are using two distractors at same distractor location in elevation trials

In [3]:

target_azim = 0
elevations = [-20, 0, 40]

# distractor_azims = [0, 10, 30] 
distractor_azims = [0, 5, 10, 20, 30, 40, 60, 90, 120, 150, 180] # Add 30, 60, 90 for comparison to our human experiments - test script handles l/r positioning 


distractor_elevation_deltas = [0, 20, 40]

snrs = np.linspace(6,-21,num=10)
room_ixs = [0]

new_min_rev_dir = Path("/om2/user/imgriff/spatial_audio_pipeline/assets/brir/mit_bldg46room1004_min_reverb")
room_manifest = pd.read_pickle(new_min_rev_dir / 'manifest_room.pdpkl')

def get_room_str(room_ix):
    return  (new_min_rev_dir / f"room{room_ix:04}.hdf5").as_posix()

def get_room_meta_dict(room_ix):
    manifest_path = (new_min_rev_dir / "manifest_brir.pdpkl").as_posix()
    index_room = room_ix
    h5_fn = get_room_str(room_ix)
    return dict(room_manifest=manifest_path, index_room=index_room, h5_fn=h5_fn)

# create dict of trial dicts for azimuth experiment 

azim_cond_dict = {ix:{'target_loc': (0, elev),
                      'distract_loc': [dist_az, elev],
                      'snr': snr,
                      'symmetric_distractor': True,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (elev, dist_az, snr, room_ix) in
              enumerate(itertools.product(*[elevations, distractor_azims, snrs, room_ixs]))} 

n_azim_conds = len(azim_cond_dict)
# create dict of trial dicts for elevation experiment 
elev_cond_dict = {n_azim_conds + ix :{'target_loc': (0, elev),
                      'distract_loc': [0, elev - (np.sign(elev) * elev_delta)],
                      'snr': snr,
                      'symmetric_distractor': True,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (elev, elev_delta, snr, room_ix) in
              enumerate(itertools.product(*[elevations, distractor_elevation_deltas, snrs, room_ixs]))} 


all_cond_dict = {**azim_cond_dict, **elev_cond_dict}
# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))
print(len(all_cond_dict))

420


In [4]:
# all_conds
# all conditions 
# with open('binaural_test_manifests/symmetric_distractor_conditions_w_front_back_neg_21_to_6_dBSNR_min_reverb_mit_room.pkl', 'wb') as f:
with open('binaural_test_manifests/human_array_exmpt_sim_w_front_back_21_to_6_dBSNR_min_reverb_mit_room.pkl', 'wb') as f:
    pickle.dump(all_cond_dict, f)

#### Manifest to re-run azimuth thresholds in new room

In [5]:

target_azim = 0
elevations = [-20, 0, 40]

# distractor_azims = [0, 10, 30] 
distractor_azims = [0, 5, 10, 20, 30, 40, 60, 90, 120, 150, 180] # Add 30, 60, 90 for comparison to our human experiments - test script handles l/r positioning 


distractor_elevation_deltas = [0, 20, 40]

snrs = np.linspace(6,-21,num=10)
room_ixs = [0]

new_min_rev_dir = Path("/om2/user/imgriff/spatial_audio_pipeline/assets/brir/mit_bldg46room1004_min_reverb")
room_manifest = pd.read_pickle(new_min_rev_dir / 'manifest_room.pdpkl')

def get_room_str(room_ix):
    return  (new_min_rev_dir / f"room{room_ix:04}.hdf5").as_posix()

def get_room_meta_dict(room_ix):
    manifest_path = (new_min_rev_dir / "manifest_brir.pdpkl").as_posix()
    index_room = room_ix
    h5_fn = get_room_str(room_ix)
    return dict(room_manifest=manifest_path, index_room=index_room, h5_fn=h5_fn)

# create dict of trial dicts for azimuth experiment 

azim_cond_dict = {ix:{'target_loc': (0, elev),
                      'distract_loc': [dist_az, elev],
                      'snr': snr,
                      'symmetric_distractor': True,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (elev, dist_az, snr, room_ix) in
              enumerate(itertools.product(*[elevations, distractor_azims, snrs, room_ixs]))} 

n_azim_conds = len(azim_cond_dict)
# create dict of trial dicts for elevation experiment 
elev_cond_dict = {n_azim_conds + ix :{'target_loc': (0, elev),
                      'distract_loc': [0, elev - (np.sign(elev) * elev_delta)],
                      'snr': snr,
                      'symmetric_distractor': False,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (elev, elev_delta, snr, room_ix) in
              enumerate(itertools.product(*[elevations, distractor_elevation_deltas, snrs, room_ixs]))} 


all_cond_dict = {**azim_cond_dict, **elev_cond_dict}
# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))
print(len(all_cond_dict))

420


### Add anechoic room 

In [6]:

target_azim = 0
elevations = [-20, 0, 40]

# distractor_azims = [0, 10, 30] 
distractor_azims = [0, 5, 10, 20, 30, 40, 60, 90, 120, 150, 180] # Add 30, 60, 90 for comparison to our human experiments - test script handles l/r positioning 


distractor_elevation_deltas = [0, 20, 40]

snrs = np.linspace(6,-21, num=10)
room_ixs = [0]

def get_room_str(room_ix):
    return f"/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/room{room_ix:04}.hdf5"

def get_room_meta_dict(room_ix):
    manifest_path = "/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/manifest_brir.pdpkl"
    index_room = room_ix
    h5_fn = get_room_str(room_ix)
    return dict(room_manifest=manifest_path, index_room=index_room, h5_fn=h5_fn)


# create dict of trial dicts for azimuth experiment 
n_existing = len(all_cond_dict)
azim_cond_dict = {ix + n_existing :{'target_loc': (0, elev),
                      'distract_loc': [dist_az, elev],
                      'snr': snr,
                      'symmetric_distractor': True,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (elev, dist_az, snr, room_ix) in
              enumerate(itertools.product(*[elevations, distractor_azims, snrs, room_ixs]))} 

n_azim_conds = len(azim_cond_dict) + n_existing
# create dict of trial dicts for elevation experiment 
elev_cond_dict = {n_azim_conds + ix :{'target_loc': (0, elev),
                      'distract_loc': [0, elev - (np.sign(elev) * elev_delta)],
                      'snr': snr,
                      'symmetric_distractor': False,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (elev, elev_delta, snr, room_ix) in
              enumerate(itertools.product(*[elevations, distractor_elevation_deltas, snrs, room_ixs]))} 


all_cond_dict = {**all_cond_dict, **azim_cond_dict, **elev_cond_dict}
# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))
print(len(all_cond_dict))

840


### Add standard speaker array

In [7]:

target_azim = 0
elevations = [-20, 0, 40]

# distractor_azims = [0, 10, 30] 
distractor_azims = [0, 5, 10, 20, 30, 40, 60, 90, 120, 150, 180] # Add 30, 60, 90 for comparison to our human experiments - test script handles l/r positioning 


distractor_elevation_deltas = [0, 20, 40]

snrs = np.linspace(6,-21,num=10)
room_ixs = [0]

def get_room_str(room_ix):
    return f"/om2/user/msaddler/spatial_audio_pipeline/assets/brir/mit_bldg46room1004/room{room_ix:04}.hdf5"

def get_room_meta_dict(room_ix):
    manifest_path = "/om2/user/msaddler/spatial_audio_pipeline/assets/brir/mit_bldg46room1004/manifest_brir.pdpkl"
    index_room = room_ix
    h5_fn = get_room_str(room_ix)
    return dict(room_manifest=manifest_path, index_room=index_room, h5_fn=h5_fn)


# create dict of trial dicts for azimuth experiment 
n_existing = len(all_cond_dict)
azim_cond_dict = {ix + n_existing :{'target_loc': (0, elev),
                      'distract_loc': [dist_az, elev],
                      'snr': snr,
                      'symmetric_distractor': True,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (elev, dist_az, snr, room_ix) in
              enumerate(itertools.product(*[elevations, distractor_azims, snrs, room_ixs]))} 

n_azim_conds = len(azim_cond_dict) + n_existing
# create dict of trial dicts for elevation experiment 
elev_cond_dict = {n_azim_conds + ix :{'target_loc': (0, elev),
                      'distract_loc': [0, elev - (np.sign(elev) * elev_delta)],
                      'snr': snr,
                      'symmetric_distractor': False,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (elev, elev_delta, snr, room_ix) in
              enumerate(itertools.product(*[elevations, distractor_elevation_deltas, snrs, room_ixs]))} 


all_cond_dict = {**all_cond_dict, **azim_cond_dict, **elev_cond_dict}
# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))
print(len(all_cond_dict))

1260


In [8]:
len(all_cond_dict)

1260

In [9]:
all_cond_dict[0]

{'target_loc': (0, -20),
 'distract_loc': [0, -20],
 'snr': 6.0,
 'symmetric_distractor': True,
 'test_room_meta': {'room_manifest': '/om2/user/imgriff/spatial_audio_pipeline/assets/brir/mit_bldg46room1004_min_reverb/manifest_brir.pdpkl',
  'index_room': 0,
  'h5_fn': '/om2/user/imgriff/spatial_audio_pipeline/assets/brir/mit_bldg46room1004_min_reverb/room0000.hdf5'}}

In [10]:
# all_conds
# all conditions 
# with open('binaural_test_manifests/symmetric_distractor_conditions_w_front_back_neg_21_to_6_dBSNR_min_reverb_mit_room.pkl', 'wb') as f:
with open('binaural_test_manifests/symmetric_distractor_conditions_w_front_back_neg_21_to_6_dBSNR_test_rooms.pkl', 'wb') as f:
    pickle.dump(all_cond_dict, f)

## Make sanity check 2 distractor in elevation condition

In [12]:

target_azim = 0
elevations = [-20, 0, 40]

# distractor_azims = [0, 10, 30] 


distractor_elevation_deltas = [0, 20, 40]

snrs = np.linspace(6,-21,num=10)
room_ixs = [0]

new_min_rev_dir = Path("/om2/user/imgriff/spatial_audio_pipeline/assets/brir/mit_bldg46room1004_min_reverb")
room_manifest = pd.read_pickle(new_min_rev_dir / 'manifest_room.pdpkl')

def get_room_str(room_ix):
    return  (new_min_rev_dir / f"room{room_ix:04}.hdf5").as_posix()

def get_room_meta_dict(room_ix):
    manifest_path = (new_min_rev_dir / "manifest_brir.pdpkl").as_posix()
    index_room = room_ix
    h5_fn = get_room_str(room_ix)
    return dict(room_manifest=manifest_path, index_room=index_room, h5_fn=h5_fn)

# create dict of trial dicts for elevation experiment 
elev_cond_dict = {ix :{'target_loc': (0, elev),
                      'distract_loc': [0, elev - (np.sign(elev) * elev_delta)],
                      'snr': snr,
                      'symmetric_distractor': True,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (elev, elev_delta, snr, room_ix) in
              enumerate(itertools.product(*[elevations, distractor_elevation_deltas, snrs, room_ixs]))} 


elev_cond_dict# Assert keys contiguous
assert np.all(np.array(list(elev_cond_dict.keys())) == np.arange(len(elev_cond_dict)))
print(len(elev_cond_dict))

90


In [13]:
with open('binaural_test_manifests/symmetric_distractor_in_elev_neg_21_to_6_dBSNR_min_reverb_room.pkl', 'wb') as f:
    pickle.dump(elev_cond_dict, f)